<a href="https://colab.research.google.com/github/chewanna7-code/NatureInsightStudy/blob/main/Observed_Rainfall_Flow_and_Modelled_Hydrograph_Comparison_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Observed Rainfall, River Flow and Modelled Hydrograph Comparison

This notebook compares observed rainfall and river flow records against modelled hydrograph outputs for a selected catchment. The flow 15 data shows flow in smaller temporal increments than NRFA. These datasets are very large to handle, especially with more observed measurements. Use of this code allows display for a more defined period, so observations can be made. Often, the file is too large to view dates, therefore, this code was developed.

It is designed as a reusable workflow for any catchment where equivalent datasets are available. In the dissertation context, it was used to compare observed flood responses with NatureInsight® / SCALGO hydrograph outputs to assess how modelled design-event behaviour compares with real recorded flood events, in terms of peak flow, and rainfall.

The notebook supports:

- importing observed rainfall data;
- importing observed river flow data;
- importing a modelled hydrograph export;
- selecting flood-event windows for comparison;
- summarising antecedent rainfall and peak flow response;
- producing figures for dissertation reporting or appendices.

## Data required

You need three inputs:

| Input | Purpose | Typical source |
|---|---|---|
| Daily rainfall CSV | Shows rainfall before and during the event | Gauge or gridded rainfall export, e.g. NFRA rainfall record |
| River flow CSV | Shows observed flow response | NRFA, Flow15, or equivalent gauge flow record |
| Modelled hydrograph Excel | Provides modelled event hydrograph or return-period outputs | NatureInsight® / SCALGO export or equivalent model output |

The workflow is intentionally general. Rename variables such as catchment name, gauge name and event dates to suit your own study area.


## 1. Import libraries

This cell imports the packages used for data handling, Excel reading and plotting. It also defines a consistent figure style for export-ready plots, althoguh this was more for the purpose of my dissertation.


In [ ]:
import io
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.colab import files
from openpyxl import load_workbook

plt.rcParams["font.family"] = "DejaVu Serif"


## 2. Set catchment and analysis information

Edit this section before running the rest of the notebook.

These values are used in figure titles, labels and notes. Keeping them in one place makes the workflow easier to reuse for another catchment.

### Guidance for selecting event dates

Choose event windows that include more than just the peak day. A useful window should usually capture:

- several days of antecedent rainfall before the peak;
- the main rainfall period;
- the flow peak;
- part of the recession limb after the peak.

This matters because literature revelaed that catchment response is often affected by antecedent rainfall. If the catchment was already saturated before the main event, runoff response may be faster and peak flows may be higher. A wider window helps show whether the flood peak was caused by a single rainfall event or by accumulated wet conditions. It is hugely beneficial in contextualising the flood-associated risks of specific catchments.


In [ ]:
# User inputs
# Edit these values for the catchment or gauge being analysed.

CATCHMENT_NAME = "Example Catchment"
GAUGE_NAME = "Example Gauge"
FLOW_SOURCE_NOTE = "Observed flow data"
RAINFALL_SOURCE_NOTE = "Observed rainfall data"
MODEL_SOURCE_NOTE = "Modelled hydrograph export"

# Optional modelled design rainfall depth shown on rainfall plots.
# Leave as None if not required.
DESIGN_STORM_DEPTH_MM = None

# Optional modelled return-period peak flow lines, obtained from natureinsight modelling.
# Format: (flow_value_m3s, label)
# Leave as an empty list if not required.
MODEL_PEAK_LINES = [
    # (260.31, "RP10"),
    # (360.84, "RP50"),
    # (409.37, "RP100"),
    # (461.72, "RP200"),
]

# List of observed events to compare.
# Each event must include a name, start date, end date and approximate peak date.
# using a notable flood in an area acts as a good comparison point e.g 10 days either side of Morpeth 2008 flood.
EVENTS = [
    {
        "name": "Example Flood Event",
        "start": "YYYY-MM-DD",
        "end": "YYYY-MM-DD",
        "peak_date": "YYYY-MM-DD",
        "note": "Brief contextual note explaining why this event was selected."
    }
]

## 3. Upload and read observed rainfall data

Upload a rainfall CSV containing a date column and a rainfall depth column. I recomend NFRA rainfall data, as it can be compared directly with gauging station location.

The loader is flexible and will try to identify common column names. If your file has unusual headings, edit `date_column` and `rainfall_column` after upload.

Expected output dataframe:

| column | meaning |
|---|---|
| `date` | daily rainfall date |
| `rainfall_mm` | rainfall depth in mm |


In [ ]:
def read_rainfall_csv(uploaded_bytes, date_column=None, rainfall_column=None):
    """Read rainfall CSV and standardise columns to date and rainfall_mm."""

    df = pd.read_csv(io.BytesIO(uploaded_bytes))

    # Try to identify date and rainfall columns automatically.
    lower_cols = {c.lower().strip(): c for c in df.columns}

    if date_column is None:
        date_candidates = ["date", "data", "datetime", "time"]
        date_column = next((lower_cols[c] for c in date_candidates if c in lower_cols), df.columns[0])

    if rainfall_column is None:
        rain_candidates = ["rainfall_mm", "rainfall", "rain", "last", "value"]
        rainfall_column = next((lower_cols[c] for c in rain_candidates if c in lower_cols), df.columns[1])

    df = df[[date_column, rainfall_column]].copy()
    df.columns = ["date", "rainfall_mm"]

    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
    df["rainfall_mm"] = pd.to_numeric(df["rainfall_mm"], errors="coerce")

    df = (
        df.dropna(subset=["date", "rainfall_mm"])
          .sort_values("date")
          .reset_index(drop=True)
    )

    return df


print("Upload rainfall CSV")
rain_upload = files.upload()
rain_file = list(rain_upload.keys())[0]

df_rain = read_rainfall_csv(rain_upload[rain_file])

print(f"Loaded rainfall file: {rain_file}")
print(f"Records: {len(df_rain):,}")
print(f"Date range: {df_rain['date'].min().date()} to {df_rain['date'].max().date()}")

df_rain.head()

## 4. Upload and read observed river flow data

Upload a river flow CSV containing a datetime column and a flow value column.

For NRFA Flow15 data, the file often already contains `datetime` and `value` columns. The loader standardises these to:

| column | meaning |
|---|---|
| `datetime` | flow timestamp |
| `flow_m3s` | river flow in m³/s |


In [ ]:
def read_flow_csv(uploaded_bytes, datetime_column=None, flow_column=None):
    """Read observed flow CSV and standardise columns to datetime and flow_m3s."""

    df = pd.read_csv(io.BytesIO(uploaded_bytes))

    lower_cols = {c.lower().strip(): c for c in df.columns}

    if datetime_column is None:
        datetime_candidates = ["datetime", "date time", "date", "time"]
        datetime_column = next((lower_cols[c] for c in datetime_candidates if c in lower_cols), df.columns[0])

    if flow_column is None:
        flow_candidates = ["flow_m3s", "flow", "value", "discharge", "q"]
        flow_column = next((lower_cols[c] for c in flow_candidates if c in lower_cols), df.columns[1])

    df = df[[datetime_column, flow_column]].copy()
    df.columns = ["datetime", "flow_m3s"]

    df["datetime"] = pd.to_datetime(df["datetime"], dayfirst=True, errors="coerce")
    df["flow_m3s"] = pd.to_numeric(df["flow_m3s"], errors="coerce")

    df = (
        df.dropna(subset=["datetime", "flow_m3s"])
          .sort_values("datetime")
          .reset_index(drop=True)
    )

    return df


print("Upload observed river flow CSV")
flow_upload = files.upload()
flow_file = list(flow_upload.keys())[0]

df_flow = read_flow_csv(flow_upload[flow_file])

print(f"Loaded flow file: {flow_file}")
print(f"Records: {len(df_flow):,}")
print(f"Date range: {df_flow['datetime'].min().date()} to {df_flow['datetime'].max().date()}")

df_flow.head()

## 5. Upload and read modelled hydrograph data

Upload the NatureInsight modelled hydrograph Excel export, for a select event and scenario. For my study, Rp100 was used, for greater consistency with industry, and a more general scenario.

The function searches the workbook for columns containing total time and total flow. This was designed for NatureInsight® / SCALGO exports but can be adapted for other model outputs.

The modelled hydrograph is mainly used here for visual comparison of hydrograph shape. This does not prove model validation, but it helps show whether the modelled response is broadly similar to, or different from, observed event behaviour.


In [ ]:
def read_modelled_hydrograph_excel(uploaded_bytes):
    """Extract modelled hydrograph time and flow series from an Excel workbook."""

    workbook = load_workbook(io.BytesIO(uploaded_bytes), data_only=True)
    worksheet = workbook.active

    time_col = None
    flow_col = None
    header_row = None

    # Search for likely time and flow columns.
    for row in worksheet.iter_rows():
        for cell in row:
            value = str(cell.value).strip().lower() if cell.value is not None else ""

            if "total" in value and "time" in value and ("sec" in value or "s" in value):
                time_col = cell.column
                header_row = cell.row

            if "total" in value and "flow" in value and "time" not in value:
                flow_col = cell.column

        if time_col is not None and flow_col is not None:
            break

    if time_col is None or flow_col is None or header_row is None:
        raise ValueError("Could not identify modelled time and flow columns. Check the Excel export structure.")

    times_hours = []
    flows_m3s = []

    for row_idx in range(header_row + 1, worksheet.max_row + 1):
        time_value = worksheet.cell(row_idx, time_col).value
        flow_value = worksheet.cell(row_idx, flow_col).value

        if time_value is None or flow_value is None:
            continue

        try:
            times_hours.append(float(time_value) / 3600)
            flows_m3s.append(float(flow_value))
        except ValueError:
            continue

    model_df = pd.DataFrame({
        "time_hours": times_hours,
        "flow_m3s": flows_m3s
    })

    return model_df


print("Upload modelled hydrograph Excel file")
model_upload = files.upload()
model_file = list(model_upload.keys())[0]

df_model = read_modelled_hydrograph_excel(model_upload[model_file])

print(f"Loaded modelled hydrograph file: {model_file}")
print(f"Points: {len(df_model):,}")
print(f"Modelled peak: {df_model['flow_m3s'].max():.2f} m³/s")

df_model.head()

## 6. Summarise observed events

This section calculates event-level values for each selected date window:

- observed peak flow;
- timing of observed peak flow;
- rainfall before the selected peak date;
- rainfall on the selected peak date.

The antecedent rainfall value is a simple sum of rainfall before the peak date within the selected event window. It is not a full soil moisture model, but it provides a useful indication of pre-event wetness, emphaisisng the revelvance of event context.


In [ ]:
def summarise_event(event, df_flow, df_rain):
    """Summarise rainfall and flow conditions within one event window."""

    start = pd.to_datetime(event["start"])
    end = pd.to_datetime(event["end"])
    peak_date = pd.to_datetime(event["peak_date"])

    event_flow = df_flow[
        (df_flow["datetime"] >= start) &
        (df_flow["datetime"] <= end)
    ].copy()

    event_rain = df_rain[
        (df_rain["date"] >= start) &
        (df_rain["date"] <= end)
    ].copy()

    if event_flow.empty:
        return {
            "Event": event["name"],
            "Start": start,
            "End": end,
            "Observed Peak Flow (m³/s)": np.nan,
            "Observed Peak Time": pd.NaT,
            "Antecedent Rainfall Before Peak (mm)": event_rain[event_rain["date"] < peak_date]["rainfall_mm"].sum(),
            "Peak Day Rainfall (mm)": np.nan,
            "Note": event.get("note", "")
        }

    peak_idx = event_flow["flow_m3s"].idxmax()
    peak_flow = event_flow.loc[peak_idx, "flow_m3s"]
    peak_time = event_flow.loc[peak_idx, "datetime"]

    antecedent_rain = event_rain[event_rain["date"] < peak_date]["rainfall_mm"].sum()

    peak_day_rain = event_rain.loc[
        event_rain["date"] == peak_date,
        "rainfall_mm"
    ].sum()

    return {
        "Event": event["name"],
        "Start": start.date(),
        "End": end.date(),
        "Observed Peak Flow (m³/s)": peak_flow,
        "Observed Peak Time": peak_time,
        "Antecedent Rainfall Before Peak (mm)": antecedent_rain,
        "Peak Day Rainfall (mm)": peak_day_rain,
        "Note": event.get("note", "")
    }


event_summary = pd.DataFrame([
    summarise_event(event, df_flow, df_rain)
    for event in EVENTS
])

event_summary

## 7. Plot rainfall and river flow for each event

Each figure shows rainfall above the observed hydrograph. This format helps link rainfall timing to flow response. The time periods are the same, despite measurements not being at the exact same point, but aggregation and totals ae used for comparison.

Interpretation points to consider:

- Did the peak flow happen during the main rainfall or after it?
- Was there substantial antecedent rainfall before the peak?
- Is the observed hydrograph single-peaked or multi-peaked?
- Do modelled return-period peak lines sit above, below or within observed event peaks?
- Could gauge position, catchment scale or routing explain timing differences?
- How does this compare in shape to the model, and why?


In [ ]:
def safe_filename(text):
    """Convert event names into safe output filenames."""
# Plot titles and event names often contain spaces, commas,
# brackets or other symbols that can make file saving messy.
    text = re.sub(r"[^A-Za-z0-9_ -]", "", text)  # Remove characters that may cause problems in saved filenames
    text = text.replace(" ", "_")  # Replace spaces with underscores for cleaner file naming
    return text[:120]  # Limit filename length to avoid very long saved outputs, can be adjusted if needed.

# Main plotting function
# This function creates one rainfall-flow comparison plot
# for each event listed in the EVENTS user-input section.
#
# For each selected event, the function:
# 1. Reads the event start date, end date and selected peak date
# 2. Filters observed flow data to the event window
# 3. Filters rainfall data to the same period
# 4. Identifies the maximum observed river flow within the window
# 5. Plots rainfall and river flow on separate panels
# 6. Adds optional modelled/design storm reference lines
# 7. Saves the figure as a high-resolution PNG


def plot_event(event, df_flow, df_rain):
    """Create a rainfall-flow comparison plot for one event."""
# Convert event dates into pandas datetime format
# - start: beginning of plotting window, chose one early enough to include antecedent impact.
# - end: end of plotting window
# - peak_date: key date selected by the user
    start = pd.to_datetime(event["start"])
    end = pd.to_datetime(event["end"])
    peak_date = pd.to_datetime(event["peak_date"])
# Filter observed flow data to selected event window, isolating the hydrgraph response for a chosen event period.
    event_flow = df_flow[
        (df_flow["datetime"] >= start) &
        (df_flow["datetime"] <= end)
    ].copy()
# Filter rainfall data to the same event window
    event_rain = df_rain[
        (df_rain["date"] >= start) &
        (df_rain["date"] <= end)
    ].copy()

     # Stop if no flow data is available
     # Flow data is essential for the comparison, so the plot
    # is skipped if the selected date range contains no flow data.

    if event_flow.empty:
        print(f"No flow data available for {event['name']}")
        return None
 # Identify observed peak flow within the event window
    peak_idx = event_flow["flow_m3s"].idxmax()
    peak_flow = event_flow.loc[peak_idx, "flow_m3s"]
    peak_time = event_flow.loc[peak_idx, "datetime"]

# Rainfall is separated from flow because the two use different units and scales.

    fig, (ax_rain, ax_flow) = plt.subplots(
        2, 1,
        figsize=(13, 8),
        gridspec_kw={"height_ratios": [1, 3]},
        sharex=False
    )

    fig.patch.set_facecolor("white")

    # Rainfall panel

    # Daily rainfall is shown as bars.
    #
    # Hydrology convention often displays rainfall inverted above the hydrograph so that rainfall visually appears to "fall" into the catchment.
     if not event_rain.empty:
        ax_rain.bar(
            event_rain["date"],
            event_rain["rainfall_mm"],
            width=0.8,
            label="Daily rainfall"
        )
# This can be used to compare observed daily rainfall with a modelled/design storm depth. OPTIONAL.
    if DESIGN_STORM_DEPTH_MM is not None:
        ax_rain.axhline(
            DESIGN_STORM_DEPTH_MM,
            linestyle=":",
            linewidth=1.5,
            label=f"Modelled design storm: {DESIGN_STORM_DEPTH_MM} mm"
        )
 # Mark selected peak date on rainfall panel

    # This helps show whether rainfall occurred before, during or after the selected flood peak.
    ax_rain.axvline(
        peak_date,
        linestyle="--",
        linewidth=1.5,
        label="Selected peak date"
    )

    ax_rain.set_ylabel("Rainfall (mm)")
    ax_rain.invert_yaxis()
    ax_rain.legend(fontsize=8, loc="lower right")

    # Flow panel, plotted hydrograph for oberved flow.,#
    ax_flow.plot(
        event_flow["datetime"],
        event_flow["flow_m3s"],
        linewidth=2,
        label="Observed flow"
    )

    ax_flow.scatter(
        [peak_time],
        [peak_flow],
        s=90,
        zorder=5,
        label=f"Observed peak: {peak_flow:.1f} m³/s"
    )

    ax_flow.axvline(
        peak_time,
        linestyle="--",
        linewidth=1.5
    )
 # MODEL_PEAK_LINES should be defined earlier as:
    #
    # MODEL_PEAK_LINES = [
    #     (flow_value, "label"),
    #     (flow_value, "label")
    # ]
    #
    # Example:
    # MODEL_PEAK_LINES = [
    #     (228, "2012 observed benchmark"),
    #     (335, "2008 observed benchmark")
    # ]
    #
    # These lines help visually compare observed event peaks

    for flow_value, label in MODEL_PEAK_LINES:
        ax_flow.axhline(
            flow_value,
            linestyle=":",
            linewidth=1.3,
            label=f"Modelled {label}: {flow_value:.1f} m³/s"
        )
    # Format flow plot
    ax_flow.set_xlabel("Date")
    ax_flow.set_ylabel("Flow (m³/s)")
    ax_flow.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    ax_flow.legend(fontsize=8, loc="upper left", ncol=2)

    ax_flow.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %Y"))
    ax_flow.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.setp(ax_flow.xaxis.get_majorticklabels(), rotation=30, ha="right")

# Figure title and notes
  # Title uses editable user-input metadata:
    # - catchment name
    # - gauge/station name
    # - event name

    fig.suptitle(
        f"{CATCHMENT_NAME} at {GAUGE_NAME} — {event['name']}",
        fontsize=13,
        fontweight="bold",
        x=0.06,
        ha="left"
    )
# Optional event-specific note
    # Useful for recording context such as:
    # - antecedent rainfall
    # - known flood impacts
    # - data caveats
    # - event selection reasoning
    fig.text(
        0.06,
        0.94,
        event.get("note", ""),
        fontsize=9,
        style="italic"
    )

    fig.text(
        0.06,
        0.01,
        f"Sources: {FLOW_SOURCE_NOTE}; {RAINFALL_SOURCE_NOTE}; {MODEL_SOURCE_NOTE}. Analysis: Author.",
        fontsize=8
    )

    plt.tight_layout(rect=[0, 0.03, 1, 0.93])

    output_name = f"{safe_filename(event['name'])}_rainfall_flow_comparison.png"
    plt.savefig(output_name, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()

    print(f"Saved: {output_name}")
    return output_name

# Outputs are stored in saved_figures so they can be checked,
# downloaded or referenced later.
saved_figures = []

for event in EVENTS:
    output = plot_event(event, df_flow, df_rain)

    if output is not None:
        saved_figures.append(output)

print("Event plots complete.")

## 8. Compare modelled and observed hydrograph shape

This section normalises hydrograph shapes so that modelled and observed responses can be compared visually.

This is useful because observed and modelled events may not have the same absolute peak flow. Normalising helps focus on hydrograph shape, including:

- steepness of rising limb;
- timing and sharpness of peak;
- recession behaviour;
- whether observed flow is single-peaked or multi-peaked.

This should be interpreted as a qualitative comparison rather than formal validation.


In [ ]:
def plot_hydrograph_shape_comparison(event, df_flow, df_model):
    """Compare normalised modelled hydrograph shape with one observed event."""

    start = pd.to_datetime(event["start"])
    end = pd.to_datetime(event["end"])

    observed = df_flow[
        (df_flow["datetime"] >= start) &
        (df_flow["datetime"] <= end)
    ].copy()

    if observed.empty or df_model.empty:
        print("Observed or modelled data missing. Shape comparison not created.")
        return None

    # Normalise observed flow to 0–1.
    observed_values = observed["flow_m3s"].to_numpy()
    observed_norm = (observed_values - observed_values.min()) / (observed_values.max() - observed_values.min())

    observed_hours = (
        observed["datetime"] - observed["datetime"].iloc[0]
    ).dt.total_seconds() / 3600

    # Normalise modelled flow to 0–1.
    model_values = df_model["flow_m3s"].to_numpy()
    model_norm = (model_values - model_values.min()) / (model_values.max() - model_values.min())

    fig, ax = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor("white")

    ax.plot(
        df_model["time_hours"],
        model_norm,
        linestyle="--",
        linewidth=2,
        label="Modelled hydrograph shape"
    )

    ax.plot(
        observed_hours,
        observed_norm,
        linewidth=1.8,
        label=f"Observed hydrograph shape: {event['name']}"
    )

    ax.set_xlabel("Time from event start (hours)")
    ax.set_ylabel("Normalised flow")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    ax.legend(fontsize=9)

    fig.suptitle(
        f"Modelled vs Observed Hydrograph Shape — {CATCHMENT_NAME} at {GAUGE_NAME}",
        fontsize=13,
        fontweight="bold",
        x=0.06,
        ha="left"
    )

    fig.text(
        0.06,
        0.01,
        "Normalised comparison shown for hydrograph shape only. Analysis: Author.",
        fontsize=8
    )

    plt.tight_layout(rect=[0, 0.04, 1, 0.93])

    output_name = f"{safe_filename(event['name'])}_hydrograph_shape_comparison.png"
    plt.savefig(output_name, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()

    print(f"Saved: {output_name}")
    return output_name


# Select the event used for hydrograph shape comparison.
# By default, this uses the first event in the EVENTS list.
shape_event = EVENTS[0]

shape_output = plot_hydrograph_shape_comparison(
    shape_event,
    df_flow,
    df_model
)

if shape_output is not None:
    saved_figures.append(shape_output)

## 9. Download outputs

Run this cell to download the exported figures from Colab.

The figures can then be inserted into the dissertation, appendix or GitHub repository. Use figure captions to explain that the plots provide contextual comparison rather than formal model validation.


In [ ]:
for figure in saved_figures:
    try:
        files.download(figure)
        print(f"Downloaded: {figure}")
    except Exception as error:
        print(f"Could not download {figure}: {error}")

## Notes for adapting this notebook

To reuse this notebook for another catchment:

1. Update `CATCHMENT_NAME` and `GAUGE_NAME`.
2. Upload the relevant rainfall, flow and modelled hydrograph files.
3. Edit the `EVENTS` list to match the flood events being investigated.
4. Add modelled return-period peak lines to `MODEL_PEAK_LINES` if useful.
5. Check that event windows capture antecedent rainfall, the main peak and recession period.
6. Use the event summary table to support written interpretation.

For a dissertation or report, the key discussion is not simply whether modelled and observed peaks are identical. More useful interpretation can consider whether the modelled hydrograph captures the general scale, timing and shape of observed catchment response, while recognising differences in event rainfall, antecedent wetness, routing, gauge location and model assumptions.
